# Visualization in matplotlib

## --------------------
## Block 0 — imports
## --------------------

In [ ]:

import os
from pathlib import Path
os.environ['USE_PYGEOS'] = '0'

import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import plotly.express as px
import json

In [ ]:
# -----------------------------------------------------------------------------
# Block 0 — ROOTS
# -----------------------------------------------------------------------------

In [ ]:
# Point ROOT at the folder that holds your drive
ROOT = Path("S:/")
export_path = ROOT / "the_tentative_team/results/"

PC6_POLY_GPKG = ROOT / "source/MISC/2025-cbs_pc6_2024_v1/cbs_pc6_2024_v1.gpkg"
PC4_POLY_GPKG = ROOT / "the_tentative_team/raw_data/2025-cbs_pc4_2024_v1/cbs_pc4_2024_v1.gpkg"
NUTS_POLY_GPKG = ROOT / "source/MISC/cbsgebiedsindelingen2016_heden/cbsgebiedsindelingen2024.gpkg"

KVK = ROOT / "the_tentative_team/processed_data/pc4_data_number_third_places.csv"

# Engine for all geopandas reads (fiona is broken on the secure machine).
GEO_ENGINE = "pyogrio"

# -----------------------------------------------------------------------------
# Block 1 — Data loading
# -----------------------------------------------------------------------------

In [ ]:
# Load the CSV file into a DataFrame
import os
# indexed_csv = pd.read_csv("S:/the_tentative_team/processed_data/indexed_data.csv")

# Load the GeoPackage file into a GeoDataFrame
indexed_gpkg = gpd.read_file(export_path / "indexed_data.gpkg", engine=GEO_ENGINE)

In [ ]:
# 1.2 Validation of data loading
print(indexed_gpkg.groupby("stedelijkheid")["third_places_community_ranked_by_urbanisation"].max())
print(indexed_gpkg.groupby("stedelijkheid")["third_places_community_ranked_by_urbanisation"].median())
print(indexed_gpkg.groupby("stedelijkheid")["third_places_community_ranked_by_urbanisation"].min())

# -----------------------------------------------------------------------------
# Block 2 — Visualiation + Validation
# -----------------------------------------------------------------------------

In [ ]:
# 2.1 Plotting one example from the index
# Create the figure and axis objects
fig, ax = plt.subplots(figsize=(10, 12))

# Plot the data
indexed_gpkg.plot(
    ax=ax, 
    edgecolor="white", 
    linewidth=0.05,
    column="third_places_cultural_ranked",
    cmap="viridis",  
    legend=True,
    legend_kwds={
        'label': "Third Places: cultural indexed",
        'shrink': 0.8               
    }
)
# Set the title and axis properties
ax.set_title("Third Places: cultural indexed")
ax.set_axis_off()
ax.set_aspect("equal")

In [ ]:
#2.2 Plotting & Exporting all the maps for each social & health index
import os

os.makedirs("indexed_plots", exist_ok=True)
columns_to_plot = [c for c in indexed_gpkg.columns if c.startswith("third_places_")]
cmaps = ["viridis", "plasma", "OrRd", "Blues", "Greens"]

for i, col in enumerate(columns_to_plot):
    fig, ax = plt.subplots(figsize = (10, 12))
    indexed_gpkg.plot(
            ax=ax, 
        edgecolor = "white", 
            linewidth = 0.05,
            column = col,
        cmap=cmaps[i %len(cmaps)],
        legend = True,
            legend_kwds={'label': col.replace("_", " ").title(),
                        'shrink': 0.6 }
    )
    title = col.replace("_", " ").title()
    ax.set_title(title)
    ax.set_axis_off()
    ax.set_aspect("equal")
    plt.savefig(f"indexed_plots/{title}.png", bbox_inches="tight", dpi=150)
    plt.close(fig)

## (Optional) Folium usage for interactive maps

In [ ]:
import folium
import pandas as pd

In [ ]:
def get_color(value):
    # Here you can define your own color mapping logic
    # For simplicity, we'll use a linear scale with 5 colors: green to red
    min_value = indexed_gpkg['third_places_cultural_ranked'].min()
    max_value = indexed_gpkg['third_places_cultural_ranked'].max()
    
    if value is None:
        return 'gray'
    
    colors = [
        '#00FF00',  # Green
        '#00CC33',
        '#009933',
        '#996633',
        '#FF0000'   # Red
    ]
    
    return colors[int(normalized_value * len(colors))]

In [ ]:
# Create a Folium map centered around the centroid of the GeoDataFrame
centroid = indexed_gpkg.geometry.centroid.iloc[0]
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=12)

# Add a choropleth layer to the map
folium.Choropleth(
    geo_data=indexed_gpkg,
    name='Third Places: Cultural Indexed',
    data=indexed_gpkg,
    columns=['pc4', 'third_places_cultural_ranked'],  # Assuming 'pc4' is a unique identifier for each feature
    key_on='feature.properties.pc4',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Third Places: Cultural Indexed'
).add_to(m)

geojson_layer = folium.GeoJson(
    indexed_gpkg,
).add_to(m)

# Add layer control to toggle layers
folium.LayerControl().add_to(m)

m